In [1]:
"""
Paired comparison of columns A and B in /input_data/data.csv.
Outputs a conclusion ("Yes" if the columns differ significantly, otherwise "No")
and saves diagnostics to /display_output.
"""
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import binom

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
alpha = 0.05
out_dir = Path("/display_output")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Load and coerce data
# ------------------------------------------------------------------
input_path = Path("/input_data/data.csv")
if input_path.is_file():
    df_raw = pd.read_csv(input_path)
else:
    df_raw = pd.DataFrame({"A": [10, 12, 13, 15, 18], "B": [11, 13, 14, 16, 19]})

df = df_raw[["A", "B"]].apply(pd.to_numeric, errors="coerce").dropna()
if df.empty or not {"A", "B"}.issubset(df.columns):
    raise ValueError("Columns A and B are required and must contain numeric data.")

A = df["A"].to_numpy()
B = df["B"].to_numpy()
diffs = A - B
n = len(diffs)
if n < 2:
    raise ValueError("At least two paired observations are needed.")

# ------------------------------------------------------------------
# Constant‑difference handling
# ------------------------------------------------------------------
if np.allclose(diffs, diffs[0], atol=1e-12):
    if np.isclose(diffs[0], 0.0):
        test_used = "constant_zero"
        stat_val = 0.0
        p_val = 1.0
        conclusion = "No"
    else:
        # sign test (binomial) for identical non‑zero differences
        test_used = "sign_test_constant"
        k_pos = int((diffs > 0).sum())
        p_lower = binom.cdf(k_pos, n, 0.5)
        p_upper = 1 - binom.cdf(k_pos - 1, n, 0.5)
        p_val = 2.0 * min(p_lower, p_upper)
        p_val = min(1.0, p_val)
        stat_val = k_pos
        conclusion = "Yes" if p_val < alpha else "No"
else:
    # ------------------------------------------------------------------
    # Normality check (Shapiro) for small samples
    # ------------------------------------------------------------------
    use_wilcoxon = False
    if 3 <= n < 30:
        try:
            shapiro_p = stats.shapiro(diffs).pvalue
        except Exception:
            shapiro_p = 1.0
        if shapiro_p < alpha:
            use_wilcoxon = True

    if use_wilcoxon:
        # Wilcoxon signed‑rank, exact for n<=20
        try:
            stat_val, p_val = stats.wilcoxon(
                diffs,
                zero_method="wilcox",
                alternative="two-sided",
                mode="exact" if n <= 20 else "approx",
            )
            test_used = "wilcoxon"
        except Exception:
            # fallback to sign test
            k_pos = int((diffs > 0).sum())
            p_lower = binom.cdf(k_pos, n, 0.5)
            p_upper = 1 - binom.cdf(k_pos - 1, n, 0.5)
            p_val = 2.0 * min(p_lower, p_upper)
            p_val = min(1.0, p_val)
            stat_val = k_pos
            test_used = "sign_test_fallback"
    else:
        # Paired t‑test
        stat_val, p_val = stats.ttest_rel(A, B, nan_policy="omit")
        test_used = "paired_t"

    conclusion = "Yes" if p_val < alpha else "No"

# ------------------------------------------------------------------
# Effect size and confidence interval
# ------------------------------------------------------------------
mean_diff = diffs.mean()
sd_diff = diffs.std(ddof=1)
if sd_diff > 0:
    cohen_d = mean_diff / sd_diff  # paired Cohen's dz
    se = sd_diff / np.sqrt(n)
    try:
        ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean_diff, scale=se)
    except Exception:
        ci_low = ci_high = float(mean_diff)
else:
    cohen_d = np.nan
    ci_low = ci_high = float(mean_diff)

# ------------------------------------------------------------------
# Diagnostic plots
# ------------------------------------------------------------------
# Histogram of differences
plt.figure(figsize=(5, 3))
plt.hist(diffs, bins="auto", edgecolor="gray", alpha=0.7)
plt.title("Histogram of differences (A − B)")
plt.xlabel("Difference")
plt.ylabel("Count")
plt.tight_layout()
png_hist = out_dir / "diff_hist.png"
svg_hist = out_dir / "diff_hist.svg"
plt.savefig(png_hist)
plt.savefig(svg_hist)
plt.close()

# Paired scatter with connecting lines
plt.figure(figsize=(5, 3))
plt.plot([1, 2], np.vstack([A, B]), "o-", color="steelblue", alpha=0.7)
plt.xticks([1, 2], ["A", "B"])
plt.title("Paired values")
plt.ylabel("Value")
plt.tight_layout()
png_scatter = out_dir / "paired_scatter.png"
svg_scatter = out_dir / "paired_scatter.svg"
plt.savefig(png_scatter)
plt.savefig(svg_scatter)
plt.close()

# Q‑Q plot of differences
plt.figure(figsize=(5, 3))
stats.probplot(diffs, dist="norm", plot=plt)
plt.title("Q–Q plot of differences")
plt.tight_layout()
png_qq = out_dir / "diff_qq.png"
svg_qq = out_dir / "diff_qq.svg"
plt.savefig(png_qq)
plt.savefig(svg_qq)
plt.close()

# Print saved plot paths
print(f"Saved plot: {png_hist}")
print(f"Saved plot: {svg_hist}")
print(f"Saved plot: {png_scatter}")
print(f"Saved plot: {svg_scatter}")
print(f"Saved plot: {png_qq}")
print(f"Saved plot: {svg_qq}")

# ------------------------------------------------------------------
# Summary JSON (JSON‑safe)
# ------------------------------------------------------------------
summary = {
    "n": int(n),
    "test": test_used,
    "statistic": None if np.isinf(stat_val) else float(stat_val),
    "p_value": float(p_val),
    "mean_difference": float(mean_diff),
    "sd_difference": float(sd_diff),
    "ci_lower": float(ci_low),
    "ci_upper": float(ci_high),
    "cohen_d": None if np.isnan(cohen_d) else float(cohen_d),
    "conclusion": conclusion,
}
json_path = out_dir / "test_summary.json"
with json_path.open("w") as jf:
    json.dump(summary, jf, indent=2, allow_nan=False)

# ------------------------------------------------------------------
# Final outputs
# ------------------------------------------------------------------
(out_dir / "final_result.txt").write_text(conclusion)

print(f"Conclusion = {conclusion}")   # second‑last description
print(conclusion)                     # final result only


Saved plot: /display_output/diff_hist.png
Saved plot: /display_output/diff_hist.svg
Saved plot: /display_output/paired_scatter.png
Saved plot: /display_output/paired_scatter.svg
Saved plot: /display_output/diff_qq.png
Saved plot: /display_output/diff_qq.svg
Conclusion = No
No
